# 03 Model Training Walkthrough

This notebook demonstrates the model initialization, loss definition, optimizer, and standard training loops.

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim

# Add project root to path
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(''))))
from src.config import get_config
from src.vision_encoder import GroundedVisionEncoder
from src.data_loader import get_dataloaders
from src.preprocessing import get_train_transforms, get_eval_transforms

In [ ]:
# Master Config
config = get_config()
config.training.batch_size = 4
config.training.num_epochs = 1

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Load dataloaders
train_transform = get_train_transforms(config.vision.input_size)
eval_transform = get_eval_transforms(config.vision.input_size)
train_loader, val_loader, _ = get_dataloaders(config, train_transform, eval_transform)
print(f"Loaded train loader with {len(train_loader)} batches.")

In [ ]:
# Initialize encoder
model = GroundedVisionEncoder(config=config.vision)
model.to(device)
print("Model loaded successfully.")

In [ ]:
# Define loss optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
print(optimizer)

In [ ]:
# Single-batch training pass demo
model.train()
batch = next(iter(train_loader))
images = batch['image'].to(device)
labels = batch['labels'].clamp(min=0.0).to(device)  # Replace -1 with 0

output = model(images)
loss = criterion(output['logits'], labels)
loss.backward()
optimizer.step()

print(f"Single batch training step loss: {loss.item():.4f}")